# ProtT5 Embedding Extraction for the LLPSense Benchmark

Extracts 1024-dimensional ProtT5 embeddings for the 854 unique sequences of the
LLPSense dataset, replicating `preprocess/embedding/t5.py` from the LLPSense repo
exactly: same checkpoint, same `[UZOB] -> X` substitution, same attention-masked
mean pooling over `last_hidden_state`.

**Before running:** Runtime -> Change runtime type -> **T4 GPU** (free tier is enough).

**Steps**
1. Install dependencies
2. Upload `llpsense_sequences.csv`
3. Load the model (~2.5 GB download, a few minutes)
4. Extract embeddings (~2-4 min on a T4)
5. Download `llpsense_t5.npz` and send it back

Total: about 10 minutes, most of it the model download.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.46.0,<5.0.0" sentencepiece
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
torch 2.14.1+cu130 | CUDA available: True
GPU: Tesla T4


## 2. Upload the sequence file

Run the cell, then pick `llpsense_sequences.csv`.

In [ ]:
from google.colab import files
import pandas as pd
up = files.upload()
seqs = pd.read_csv("llpsense_sequences.csv")
print(seqs.shape)
print(seqs.head(3))
assert {"SeqID", "Sequence"} <= set(seqs.columns)
print(f"{len(seqs)} sequences | {seqs.Sequence.str.len().sum():,} residues | longest {seqs.Sequence.str.len().max()}")

Saving llpsense_sequences.csv to llpsense_sequences.csv
(854, 3)
   SeqID                                           Sequence  length
0  S0000  GAGPYGPGASAAAAAAGGYGPGSGQQGPGQQGPGQQGPGQQGPGQQ...     530
1  S0001  SDKQIKNGECDKAYLDELVELHRRLMTLRERHILQQIVNLIEETGH...      79
2  S0002  VSNKGKRKHKNEDDNRASESKKPKTEDKNSAGHKPSSNRESSKQSA...     142
854 sequences | 268,298 residues | longest 1748


## 3. Load ProtT5

`Rostlab/prot_t5_xl_half_uniref50-enc` — the encoder-only half-precision model,
exactly the checkpoint LLPSense uses.

In [ ]:
import re, numpy as np, torch
from transformers import T5Tokenizer, T5EncoderModel

MODEL = "Rostlab/prot_t5_xl_half_uniref50-enc"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# tokenizer = T5Tokenizer.from_pretrained(MODEL, do_lower_case=False, use_safetensors=True)
# model = T5EncoderModel.from_pretrained(MODEL, use_safetensors=True).to(device)
tokenizer = T5Tokenizer.from_pretrained(MODEL, do_lower_case=False, legacy=True)
model = T5EncoderModel.from_pretrained(MODEL, use_safetensors=True).to(device)
model.requires_grad_(False)
model.eval()
print("embedding dim:", model.shared.embedding_dim)   # expect 1024

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.42G [00:00<?, ?B/s]

embedding dim: 1024


## 4. Extract embeddings

Sequences are sorted by length and batched so padding stays small. Pooling is the
attention-masked mean over `last_hidden_state`, matching their implementation
(the EOS token is included in the mean, as it is in theirs).

If you hit a CUDA out-of-memory error, lower `MAX_RESIDUES_PER_BATCH`.

In [ ]:
from tqdm.auto import tqdm

MAX_RESIDUES_PER_BATCH = 4000   # lower this if CUDA runs out of memory

order = seqs.Sequence.str.len().sort_values().index      # short -> long
ids_sorted = seqs.loc[order, "SeqID"].tolist()
seq_sorted = seqs.loc[order, "Sequence"].tolist()

# greedy batching by padded size
batches, cur = [], []
for sid, s in zip(ids_sorted, seq_sorted):
    trial = cur + [(sid, s)]
    if len(trial) * max(len(x[1]) for x in trial) > MAX_RESIDUES_PER_BATCH and cur:
        batches.append(cur); cur = [(sid, s)]
    else:
        cur = trial
if cur: batches.append(cur)
print(f"{len(batches)} batches")

emb = {}
for batch in tqdm(batches):
    bids = [b[0] for b in batch]
    raw  = [b[1] for b in batch]
    prepared = [" ".join(list(re.sub(r"[UZOB]", "X", s))) for s in raw]
    tok = tokenizer(prepared, add_special_tokens=True, padding="longest")
    input_ids = torch.tensor(tok["input_ids"]).to(device)
    attn      = torch.tensor(tok["attention_mask"]).to(device)
    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attn).last_hidden_state
    pooled = (out * attn[..., None]).sum(1) / attn.sum(1, keepdim=True)
    for sid, v in zip(bids, pooled.float().cpu().numpy()):
        emb[sid] = v

print("extracted:", len(emb), "| dim:", next(iter(emb.values())).shape)

74 batches


  0%|          | 0/74 [00:00<?, ?it/s]

extracted: 854 | dim: (1024,)


## 5. Save and download

In [ ]:
import numpy as np
keys = seqs.SeqID.tolist()                    # original CSV order
X = np.stack([emb[k] for k in keys]).astype(np.float32)
print("matrix:", X.shape)
assert X.shape == (len(seqs), 1024)
assert np.isfinite(X).all(), "non-finite values present"

np.savez_compressed("llpsense_t5.npz", SeqID=np.array(keys), X=X)
import os; print("file size: %.1f MB" % (os.path.getsize("llpsense_t5.npz")/1e6))

from google.colab import files
files.download("llpsense_t5.npz")

matrix: (854, 1024)
file size: 3.3 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Then

Send `llpsense_t5.npz` back and the three-regime benchmark runs on the real
LLPSense feature basis: 1024 ProtT5 dimensions + 13 environmental parameters.

**Sanity check before sending:** the printed matrix shape must be `(854, 1024)`.